# Post5G: OAI with Physical Radio Unit in R2LAB

## Prerequisites

Before starting, make sure you:

- Have completed the [Provision Resources](./01_provision.ipynb), [RAN split](./22_split.ipynb), and [TDD patterns](./20_tdd.ipynb) tutorials.
- Have a basic understanding of Python.
- Are familiar with 5G architecture and 5G NR radio.

## Tutorial Overview

In this tutorial, you will deploy a real Radio Unit (RU) inside the [R2LAB anechoic chamber](https://r2lab.inria.fr/index.md) at SophiaNode. 

### What You Will Build

The deployment consists of:
* **Core Network:** Open5GS core running on a virtual machine (VM).
* **Ran Deployment:** An OpenAirInterface (OAI) RAN split across a CU-CP, CU-UP, and DU:
  * A VM hosting the CU-CP and CU-UP functions.
  * A dedicated bare-metal server (`sopnode-f1`) hosting the DU and the 5G fronthaul connection.
* **Radio Unit (RU):** An [AW2S Jaguar](https://www.aw2s.com/solutions/rrh-rru/) RU hosted inside the chamber, connected to the DU over fiber.
* **User Equipment (UE):** Two physical 5G Quectel RM500Q-GL UEs (`qhat01` and `qhat02`), each attached to a Raspberry Pi 4 and connected to the RU.

### Architecture & Infrastructure

While previous tutorials relied entirely on virtual machines, real RU hardware requires a bare-metal DU to deliver the low-latency performance needed for radio signal processing. 

To support this setup, SophiaNode provides high-performance 5G fronthaul servers directly linked to RUs over fiber, alongside SLICES-BI virtual machines. In this scenario, you will combine both infrastructure types:

| Component | Target Environment | Hardware / Machine |
| :--- | :--- | :--- |
| **5G Core** | Virtual Machine | SLICES-BI VM |
| **CU-CP & CU-UP** | Virtual Machine | SLICES-BI VM |
| **DU & Fronthaul** | Bare-Metal | POS node (`sopnode-f1`) |

![RU topology](figures/ru.png)

> **Note:** While splitting the CU-CP, CU-UP, and DU is not strictly required for this physical RU setup, it is used here to demonstrate a realistic split-RAN deployment.

## Metamodel and Deployment Model

### Metamodel

The metamodel follows the same pattern used in previous tutorials, with one key addition: it can describe a physical Radio Unit (RU). 
* **Simulated Link:** If no RU is specified, the radio link is simulated.
* **Physical Link:** If an RU is specified, the gNB (or the DU in a split deployment) uses it for real-world radio transmission and reception.

#### Radio Unit (RU) Configuration

This tutorial uses the `jaguar` RU inside the [R2LAB anechoic chamber](https://r2lab.inria.fr/index.md) at SophiaNode. R2LAB is an open-access, remotely accessible wireless testbed providing software-defined radios (USRPs), 5G RUs, and physical UEs. You can check the [SophiaNode Resources page](https://gitlab.inria.fr/slices-ri/resources/sites/sophianode/resources) for full equipment details.

The [`Jaguar`](https://www.aw2s.com/solutions/rrh-rru/) is an AW2S 5G RU configured with the following parameters:
* **Frequency Band:** `n78`
* **Bandwidth:** 50 MHz
* **MIMO Configuration:** 2x2 MIMO
* **TDD Pattern:** `DDSUU` with a 2.5 ms period

> **Note:** Transmit attenuation is tuned specifically for the R2LAB environment. Examples of gNB configurations for R2LAB equipment are available [in this GitHub repository](https://github.com/sopnode/oai5g-rru/tree/main/rru).

#### User Equipment (UE) Configuration

The chamber houses several types of UEs, including smartphones and Quectel modules. This tutorial connects two physical Quectel RM500Q-GL devices named `qhat01` and `qhat02`. 

Because these are physical devices rather than software-emulated UEs (such as UERANSIM), the metamodel classifies them as **commercial off-the-shelf (COTS)** equipment. Their SUPI and authentication data are listed on the [R2LAB hardware page](https://r2lab.inria.fr/hardware.md).

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
rus:
  jaguar:
    type: real
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

ran:
    components:
      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          radioConfiguration:
            nrBand: 78
            channelBandwidthMhz: 50
            pointAFrequencyArfcn: 626748
            ssbArfcn: 627360
            ssbPbchBlockPowerDbm: -10
            tddConfiguration:
              periodicityMilliseconds: 2.5
              nrofDownlinkSlots: 2
              nrofDownlinkSymbols: 6
              nrofUplinkSlots: 2
              nrofUplinkSymbols: 4
          ru: jaguar
          cu_cp: cucp
          cu_up: cuup

ues:
  qhat01:
    type: cots
    config:
      supi: "001010000000006"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1

  qhat02:
    type: cots
    config:
      supi: "001010000000007"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1
""")

### Deployment Model

This tutorial combines several resource types: two VMs in SLICES-BI, one bare-metal machine in POS, one RU in R2LAB, and two physical UEs in R2LAB. These resources are managed by three distinct provisioners:

* **`bi` Provisioner:** Manages the VMs using the `fr-sophia2-bi-vm1` infrastructure, which is directly connected to R2LAB.
* **`legacy_pos` Provisioner:** Manages the POS bare-metal machine.
* **`r2lab` Provisioner:** Manages the RU and UEs hosted in R2LAB.

> **Important:** Reserve your R2LAB and POS resources *before* running this tutorial. While provisioners can attempt automatic reservations, this tutorial assumes your reservations are already active.

> **Important:** Never publish or share your R2LAB credentials.

#### Node Roles and Provisioning Parameters

* **Fronthaul Tuning:** The fronthaul machine requires performance tuning to interface correctly with the RU. This tutorial configures the Linux boot parameters for `sopnode-f1` via the node's `provisioning_params.boot_params` field.
* **RAN Distribution:** The RAN spans two machines, both assigned the `ran` role. The `node-ran` machine also carries the `cu_cp` and `cu_up` roles to host those respective functions.
* **RU & UE References:** The RU and UEs simply need to reference the `r2lab` provisioner.

#### Scheduling and Network Configuration

The CU-CP, CU-UP, and DU require specific scheduling constraints:
* Set `node_name` on the DU component to explicitly place it on `sopnode-f1`.
* Set `force_type_scheduling` on the CU-CP and CU-UP components to place each on a node with the corresponding `cu_cp` or `cu_up` role.

To establish the connection between the RU and the DU:
* Enable **Multus**, specifying the host network interface and local IP address designated for the RU connection.
* Configure OAI with Jaguar-specific settings by providing them in the DU component's `advanced_config` section.

For more details regarding SophiaNode equipment and network topology, see the [SophiaNode Resources page](https://gitlab.inria.fr/slices-ri/resources/sites/sophianode/resources).

In [2]:
! pip install python-dotenv

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

R2LAB_EMAIL = os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD = os.getenv("R2LAB_PASSWORD")
R2LAB_SLICE = os.getenv("R2LAB_SLICE")
LEGACY_POS_USER = os.getenv("LEGACY_POS_USER")

In [4]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
f"""
provisioners:
  bi-provisioner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: ru_tutorial
      create_experiment: true
      experiment_duration: 3h
      node_duration: 3h

  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: {LEGACY_POS_USER}

  r2lab:
    type: r2lab
    params:
      r2lab_slice: {R2LAB_SLICE}
      reservation_params:
        email: {R2LAB_EMAIL}
        password: {R2LAB_PASSWORD}

nodes:
  sopnode-f1:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  node-core:
    provisioner: bi-provisioner
    roles:
    - core

  node-ran:
    provisioner: bi-provisioner
    roles:
    - ran
    - cu_cp
    - cu_up

rus:
  jaguar:
    vendor: aw2s
    model: jaguar
    provisioner: r2lab

ues:
  qhat01:
    type: cots
    provisioner: r2lab

  qhat02:
    type: cots
    provisioner: r2lab

ran:
  implementation: oai
  components:
    cucp:
      type: cu_cp
      force_type_scheduling: true

    cuup:
      type: cu_up
      force_type_scheduling: true

    du1:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.102"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 1
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f1
"""
)


## Provision the Infrastructure

First, provision only the compute nodes (the virtual machines and bare-metal servers).

To ensure the physical RUs and UEs are not provisioned yet, use the `provision_nodes` function rather than `provision`. Once all compute nodes are up and ready, proceed to deploy the Kubernetes cluster.

In [5]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ran' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' fo

In [6]:
print (prov.get_ansible_inventory())

[ran_nodes]
sopnode-f1 ansible_host=sopnode-f1 ansible_port=22 ansible_user=root ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p dsaucezi@duckburg.net.cit.tum.de -p 10022"'
node-ran ansible_host=10.123.250.108 ansible_port=22 ansible_user=dsaucezi ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'

[core_nodes]
node-core ansible_host=10.123.179.15 ansible_port=22 ansible_user=dsaucezi ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHos

In [7]:
k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=False)


PLAY [node-core] ***************************************************************

PLAY [node-ran] ****************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node-ran'] *************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node-core'] ************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f1'] ***********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node-core'] ***
[WARNING]: Host 'node-ran' is using the discovered Python interpreter at '/usr

In [8]:
from pathlib import Path
DIR="state/ru"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [9]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

The infrastructure is now ready for the 5G network deployment.

## Deploy the 5G network

### Deploy the 5G core

With the infrastructure provisioned, start by deploying the 5G core.

In [10]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

TASK [helm/pr

## Tune the RAN Nodes

The Distributed Unit (DU) requires additional CPU tuning and network-interface configuration to meet connectivity and performance demands of the Radio Unit (RU). Because these settings depend on your specific RU/DU pair, they are applied directly to the machine via an **Ansible playbook**. 

The **`Miscellaneous`** manager is used to run arbitrary tasks as long as they are implemented in the form of [Ansible playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks_intro.html). The `Miscellaneous` manager executes these playbooks using the inventory automatically generated from your deployment model.

### Working with the Inventory and Playbook

* **Inspect the Inventory:** To review the generated inventory before applying changes, call `get_ansible_inventory` on the `ProvisioningManagement` manager.
* **Run the Playbook:** This tutorial uses a custom playbook tailored specifically for this hardware setup: [`misc/ansible/playbooks/tuning.yaml`](misc/ansible/playbooks/tuning.yaml).

In [11]:
print (prov.get_ansible_inventory())

[ran_nodes]
sopnode-f1 ansible_host=sopnode-f1 ansible_port=22 ansible_user=root ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p dsaucezi@duckburg.net.cit.tum.de -p 10022"'
node-ran ansible_host=10.123.250.108 ansible_port=22 ansible_user=dsaucezi ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'

[core_nodes]
node-core ansible_host=10.123.179.15 ansible_port=22 ansible_user=dsaucezi ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHos

In [12]:
from orchestrator.deployment import MiscellaneousManagement
misc = MiscellaneousManagement(preceding=core)

In [13]:
await misc.run_playbook(
    playbook="./misc/ansible/playbooks/tuning.yaml",
    roles_paths=["./misc/ansible/roles"],
)


PLAY [Setup RAN nodes] *********************************************************

TASK [Gathering Facts] *********************************************************
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1]

TASK [tuning/netplan : Upload netplan config to remote] ************************
changed: [sopnode-f1]

TASK [tuning/netplan : Generate netplan config] ********************************
changed: [sopnode-f1]

TASK [tuning/netplan : Apply netplan config] ***********************************
changed: [sopnode-f1]

TASK [tuning/netplan : Install lldpd] ******************************************
changed: [sopnode-f1]

TASK [tuning/netplan : Enable and start lldpd service] ************************

### Deploy the 5G RAN

The `sopnode-f1` machine is now configured. Provision the `jaguar` RU before deploying the DU.

In [14]:
prov_ru = ProvisioningManagement(preceding=misc)
await prov_ru.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Provision 'qhat02'] ******************************************************
TASK [Provision 'qhat01'] ******************************************************


With the radio hardware provisioned, deploy the RAN components. The CU-CP and CU-UP will run on `node-ran`, and the DU will run on `sopnode-f1`.

In [15]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=prov_ru)
await ran.deploy()

/workspaces/orchestrator/.venv-1/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='snssaiList', input_value=Slice(sst=1, sd=None, nam...maximumBitRate=None)])}), input_type=Slice])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='ru', input_value=MergedRU(name='jaguar', c...xGainDB=35, rxGainDB=75), input_type=MergedRU])
  return self.__pydantic_serializer__.to_python(



PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [sopnode-f1]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-cucp' if provided] ***
skipping: [sopnode-f1]

TASK [helm/prepare : Copy local built-in 'oai-cu_cp'

Check the pods in the `oai-ran` namespace to verify that the RAN functions are running on the expected machines.

In [16]:
print((await prov_ru.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov_ru.deployment_model.params.k8s.master_node],
    become=True, quiet=True, flatten=False))[prov_ru.deployment_model.params.k8s.master_node])

NAME                        READY   STATUS    RESTARTS   AGE   IP               NODE         NOMINATED NODE   READINESS GATES
oai-cucp-5788c66b45-hsvh4   1/1     Running   0          83s   10.244.204.120   node-ran     <none>           <none>
oai-cuup-d66c86cdb-rdvdt    1/1     Running   0          69s   10.244.139.65    node-ran     <none>           <none>
oai-du1-7854dd6b75-9m4td    1/1     Running   0          59s   10.244.214.131   sopnode-f1   <none>           <none>


Check the DU logs to confirm that it is running.

In [20]:
print ((await ran.get_logs(quiet=True, flatten=False))["du1"]["du"])

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du1
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du1
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 627360
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
        dl_subcarrierSpacing: 1
        dl_carrierBandwidth: 133
        #initialDownlinkBW

## Connect the UEs

With the 5G core network and both RAN deployments up and running, you can now provision the UEs defined in your metamodel.

For the physical R2LAB UEs, provisioning simply powers on the devices. Once active, attach them to their respective radio networks by calling `connect` on the `UEManagement` manager.

In [21]:
prov_ues = ProvisioningManagement(preceding=ran)
await prov_ues.provision()

In [23]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=prov_ues)
await ues.connect()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Connect 'qhat02'] ********************************************************

TASK [Connect 'qhat01'] ********************************************************
included: provisioning/unified_radio/ue/connect for localhost
included: provisioning/unified_radio/ue/connect for localhost

TASK [Get access token if missing] *********************************************

TASK [Get access token if missing] ************************************

Check the DU logs to confirm that the UEs are connected through the radio interface.

In [24]:
print (await ran.get_logs(quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - cucp
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
Num_Threads_PUSCH: 8
sa: 1

gNBs:
  - gNB_ID: 0x1
    gNB_name: cucp
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    tr_s_preference: "f1"

    local_s_address: "10.244.204.120"
    remote_s_address: "0.0.0.0"
    local_s_portc: "38472"
    remote_s_portc: "38472"

    # ------- SCTP definitions
    SCTP:
      SCTP_INSTREAMS: 2
      SCTP_OUTSTREAMS: 2

    # -------- AMF parameters
    amf_ip_address:
      - ipv4: 10.244.134.4

    E1_INTERFACE:
      - type: "cp"
        ipv4_cucp: "10.244.204.120"
        port_cucp: "38462"
        ipv4_cuup: "0.0.0.0"
        port_cuup: "3

To test connectivity, send a ping from each UE over its radio interface. On the QHAT devices, this interface is named `wwan0`.

Run commands directly on the UEs with the `run_ue_command` function.

In [25]:
prov_ues = ProvisioningManagement(preceding=ues)
command = ["ping", "-I", "wwan0", "-c", "10", "-D", "8.8.8.8"]
pings = await prov_ues.run_ue_command(command=command, ues=["qhat01", "qhat02"], quiet=True, flatten=False)

Ping results from `qhat01`:

In [26]:
print (pings["qhat01"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 wwan0: 56(84) bytes of data.
[1793828758.667040] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=29.1 ms
[1793828759.667445] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=28.3 ms
[1793828760.672926] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=31.8 ms
[1793828761.674259] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=32.3 ms
[1793828762.672157] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=28.8 ms
[1793828763.679165] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=33.9 ms
[1793828764.677086] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=30.9 ms
[1793828765.677486] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=29.3 ms
[1793828766.681394] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=31.8 ms
[1793828767.684299] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=32.8 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9014ms
rtt min/avg/max/mdev = 28.325/30.910/33.944/1.828 ms


Ping results from `qhat02`:

In [27]:
print (pings["qhat02"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 wwan0: 56(84) bytes of data.
[1793828759.087984] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=34.5 ms
[1793828760.083877] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=28.8 ms
[1793828761.090881] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=33.9 ms
[1793828762.081269] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=23.3 ms
[1793828763.088676] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=29.4 ms
[1793828764.088579] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=27.9 ms
[1793828765.095518] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=32.9 ms
[1793828766.093894] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=29.3 ms
[1793828767.095907] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=30.0 ms
[1793828768.098807] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=31.8 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9014ms
rtt min/avg/max/mdev = 23.339/30.179/34.534/3.139 ms


Finally, save the metamodel and deployment model to keep a record of this experiment.

In [28]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")